In [1]:
%pip install yfinance

import yfinance as yf

ticker = yf.Ticker("irs")

price = ticker.history(start="2009-02-14", end="2020-06-11")
print(price)

Note: you may need to restart the kernel to use updated packages.
                               Open      High       Low     Close  Volume  \
Date                                                                        
2009-02-17 00:00:00-05:00  1.483274  1.538041  1.460454  1.483274    5019   
2009-02-18 00:00:00-05:00  1.460455  1.510658  1.428507  1.496966   46742   
2009-02-19 00:00:00-05:00  1.460454  1.478710  1.414815  1.455890   36115   
2009-02-20 00:00:00-05:00  1.414816  1.496966  1.396560  1.442199    8266   
2009-02-23 00:00:00-05:00  1.451327  1.487839  1.437635  1.437635    9152   
...                             ...       ...       ...       ...     ...   
2020-06-04 00:00:00-04:00  2.642854  2.823048  2.642854  2.657870   50625   
2020-06-05 00:00:00-04:00  2.620329  2.905637  2.620329  2.687902  193957   
2020-06-08 00:00:00-04:00  2.717934  3.003242  2.717934  2.988226  429167   
2020-06-09 00:00:00-04:00  3.003243  3.078324  2.725443  3.010751  208106   
2020-06-10

In [2]:
#Collect Features, we want them a day behind, we give yesturdays stats to guess todays price, so score follows current day
price["Change"] = price["Close"].shift(1) - price["Open"].shift(1)
price["Score"] = (price["Close"] > price["Open"]).astype(int)
price["%Change"] = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
price["YestScore"] = price["Score"].shift(1)
price["5DayMean"] = price["Close"].rolling(5).mean().shift(1)
price["5DayVoli"] = price["Close"].rolling(5).std().shift(1)
price["5DayPerf"] = price["Score"].rolling(5).sum().shift(1)

price.dropna(inplace=True)

print(price["5DayPerf"])

Date
2009-02-24 00:00:00-05:00    2.0
2009-02-25 00:00:00-05:00    3.0
2009-02-26 00:00:00-05:00    3.0
2009-02-27 00:00:00-05:00    3.0
2009-03-02 00:00:00-05:00    2.0
                            ... 
2020-06-04 00:00:00-04:00    2.0
2020-06-05 00:00:00-04:00    3.0
2020-06-08 00:00:00-04:00    4.0
2020-06-09 00:00:00-04:00    4.0
2020-06-10 00:00:00-04:00    4.0
Name: 5DayPerf, Length: 2844, dtype: float64


In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

features = ["Change", "%Change", "CloseToOpen", "YestScore", "5DayMean", "5DayVoli", "5DayPerf"]

priceTrain = price[price.index <= "2018-06-11"]
priceTest = price[price.index > "2018-06-11"]

results = []

for i in range(len(priceTest)):
    xTrain = priceTrain[features]
    yTrain = priceTrain["Score"]

    scaler = StandardScaler()
    xTrain = scaler.fit_transform(xTrain)

    model = LogisticRegression()
    model.fit(xTrain, yTrain)

    #Grab yesturdays information to predict todays, data already transformed so i is yesturday
    yest = priceTest.iloc[[i]]
    yestScal = scaler.transform(yest[features])
    pred = model.predict(yestScal)[0]
    prob = model.predict_proba(yestScal)[0][1]
    print("Analyzing Day: ", priceTest.index[i])

    results.append({"Date": priceTest.index[i],
                    "Score": priceTest["Score"][i],
                    "Prediction": pred,
                    "Probability": prob,
                    "Open": priceTest["Open"][i],
                    "Close": priceTest["Close"][i]})
    
    priceTrain = pd.concat([priceTrain, yest])

Analyzing Day:  2018-06-12 00:00:00-04:00
Analyzing Day:  2018-06-13 00:00:00-04:00
Analyzing Day:  2018-06-14 00:00:00-04:00
Analyzing Day:  2018-06-15 00:00:00-04:00
Analyzing Day:  2018-06-18 00:00:00-04:00
Analyzing Day:  2018-06-19 00:00:00-04:00
Analyzing Day:  2018-06-20 00:00:00-04:00
Analyzing Day:  2018-06-21 00:00:00-04:00
Analyzing Day:  2018-06-22 00:00:00-04:00
Analyzing Day:  2018-06-25 00:00:00-04:00
Analyzing Day:  2018-06-26 00:00:00-04:00
Analyzing Day:  2018-06-27 00:00:00-04:00
Analyzing Day:  2018-06-28 00:00:00-04:00
Analyzing Day:  2018-06-29 00:00:00-04:00
Analyzing Day:  2018-07-02 00:00:00-04:00
Analyzing Day:  2018-07-03 00:00:00-04:00
Analyzing Day:  2018-07-05 00:00:00-04:00
Analyzing Day:  2018-07-06 00:00:00-04:00
Analyzing Day:  2018-07-09 00:00:00-04:00
Analyzing Day:  2018-07-10 00:00:00-04:00
Analyzing Day:  2018-07-11 00:00:00-04:00
Analyzing Day:  2018-07-12 00:00:00-04:00
Analyzing Day:  2018-07-13 00:00:00-04:00
Analyzing Day:  2018-07-16 00:00:0

In [7]:
results = pd.DataFrame(results)

score = results["Score"]
pred = results["Prediction"]
op = results["Open"]
close = results["Close"]

accuracy = accuracy_score(score, pred)
precision = precision_score(score, pred, zero_division=0)
recall = recall_score(score, pred, zero_division=0)
f1 = f1_score(score, pred, zero_division=0)

print("Accuracy: ", accuracy, "\nPrecision: ", precision, "\nRecall: ", recall, "\nF1: ", f1)



Accuracy:  0.49304174950298213 
Precision:  0.4625 
Recall:  0.3045267489711934 
F1:  0.36724565756823824


In [8]:
#Find out how much you would have made/lost with this method
totalIn = 0
totalOut = 0
totalInvested = 0
totalStoodOut = 0
alltradein = 0
alltradeout = 0
perftradein = 0
perftradeout = 0
for i, o, c, s in zip(pred.values, op.values, close.values, score.values):
    if i == 1:
        totalIn += 100
        totalInvested += 1
        temp = ((c - o)/o) * 100
        totalOut += 100 + temp
        print(temp, o, c)
    else:
        totalStoodOut += 1
    alltradein += 100
    alltradeout += (((c-o)/o) * 100) + 100
    if s == 1:
        perftradein += 100
        perftradeout += (((c-o)/o) * 100) + 100

print(totalIn)
print(totalOut)
print(totalInvested)
print(totalStoodOut)
print(alltradein)
print(alltradeout)
print(perftradein)
print(perftradeout)

0.288181756912372 12.162663275214861 12.197713851928711
-0.8838396106602906 11.104126016815382 11.005983352661133
-1.1334978207874722 11.132166214104446 11.005983352661133
-0.07622166570804689 9.197356142212852 9.190345764160156
2.8701009239698076 9.281477152430476 9.54786491394043
-1.705424872799825 9.48624461953551 9.324463844299316
-1.846154650789807 9.559782474284171 9.383294105529785
-0.08006527848649282 9.184744843783543 9.177391052246094
-0.39215587215335435 9.375940665169326 9.33917236328125
2.2544269146788998 9.133269477979526 9.33917236328125
2.9968491872782757 9.324464583046028 9.603904724121094
0.7309922589817551 10.059832846426897 10.133369445800781
-4.913298963840921 10.177493219560843 9.67744255065918
-2.5915994496317922 8.228767245733954 8.015510559082031
-3.7443004224850487 8.05227875189474 7.750777244567871
-4.162818014019935 7.949327581543036 7.618411540985107
-3.174601848697747 7.875789815896765 7.625764846801758
-0.09337211670360582 7.87578965118461 7.8684358596801